# Violence Video Classifier

Dataset: https://www.kaggle.com/datasets/mohamedmustafa/real-life-violence-situations-dataset

In [ ]:
!pip install safetensors -q


In [ ]:
import os, json, random, time, warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import classification_report, confusion_matrix
from safetensors.torch import save_file, load_file
from pathlib import Path
warnings.filterwarnings('ignore')


In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
DATA_ROOT = "/kaggle/input/datasets/mohamedmustafa/real-life-violence-situations-dataset/Real Life Violence Dataset"
OUTPUT_DIR  = "/kaggle/working/checkpoints"
NUM_FRAMES  = 16
IMG_SIZE    = 224
BATCH_SIZE  = 16
EPOCHS      = 20
LR          = 3e-4
DROPOUT     = 0.4
FREEZE_EP   = 3
PATIENCE    = 6
VAL_RATIO   = 0.15
SEED        = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
torch.manual_seed(SEED)
np.random.seed(SEED)


VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv"}
LABEL_MAP = {"Violence": 1, "NonViolence": 0}


In [ ]:
def collect_samples(root):
    samples = []
    for cls_name, label in LABEL_MAP.items():
        cls_dir = Path(root) / cls_name
        if not cls_dir.exists():
            for d in Path(root).iterdir():
                if d.is_dir() and d.name.lower() == cls_name.lower():
                    cls_dir = d; break
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in VIDEO_EXT:
                samples.append((str(p), label))
    return samples


In [ ]:
def split_samples(samples, val_ratio=0.15, test_ratio=0.05, seed=42):
    rng = random.Random(seed)
    v = [s for s in samples if s[1]==1]
    n = [s for s in samples if s[1]==0]
    def _split(lst):
        lst = lst.copy(); rng.shuffle(lst)
        nv = max(1, int(len(lst)*val_ratio))
        nt = max(1, int(len(lst)*test_ratio))
        return lst[nv+nt:], lst[:nv], lst[nv:nv+nt]
    tv,vv,tv2 = _split(v);  tn,vn,tn2 = _split(n)
    tr = tv+tn; rng.shuffle(tr)
    return tr, vv+vn, tv2+tn2


In [ ]:
def sample_frames(path, n=16, strategy="uniform"):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened(): return None
    total = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 1)
    idx = np.linspace(0, total-1, n, dtype=int) if strategy=="uniform" \
          else sorted(random.sample(range(total), min(n, total)))
    while len(idx) < n: idx = list(idx) + [idx[-1]]
    frames = []
    for i in idx[:n]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ret, fr = cap.read()
        frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB) if ret
                      else (frames[-1].copy() if frames else np.zeros((224,224,3),dtype=np.uint8)))
    cap.release()
    return np.stack(frames)

MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
train_tf = T.Compose([T.ToPILImage(), T.Resize(IMG_SIZE+32),
                      T.RandomCrop(IMG_SIZE), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.3,0.3,0.2,0.1), T.ToTensor(),
                      T.Normalize(MEAN, STD)])
val_tf   = T.Compose([T.ToPILImage(), T.Resize((IMG_SIZE, IMG_SIZE)),
                      T.ToTensor(), T.Normalize(MEAN, STD)])


In [ ]:
class ViolenceDataset(Dataset):
    def __init__(self, samples, num_frames=16, transform=None, strategy="uniform"):
        self.samples = samples; self.nf = num_frames
        self.transform = transform; self.strategy = strategy
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        frames = sample_frames(path, self.nf, self.strategy)
        if frames is None:
            frames = np.zeros((self.nf, 224, 224, 3), dtype=np.uint8)
        if self.transform:
            frames = torch.stack([self.transform(f) for f in frames])
        else:
            frames = torch.from_numpy(frames).permute(0,3,1,2).float()/255.
        return frames, label

samples = collect_samples(DATA_ROOT)
print(f"Total: {len(samples)}, Violence: {sum(s[1]==1 for s in samples)}, NonViolence: {sum(s[1]==0 for s in samples)}")

train_s, val_s, test_s = split_samples(samples, VAL_RATIO, seed=SEED)
print(f"Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}")

train_ds = ViolenceDataset(train_s, NUM_FRAMES, train_tf, "random")
val_ds   = ViolenceDataset(val_s,   NUM_FRAMES, val_tf,   "uniform")
test_ds  = ViolenceDataset(test_s,  NUM_FRAMES, val_tf,   "uniform")

labels_arr = np.array([s[1] for s in train_s])
w = 1.0 / np.bincount(labels_arr)[labels_arr]
sampler = WeightedRandomSampler(torch.DoubleTensor(w), len(w), replacement=True)

train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,   num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False,   num_workers=4, pin_memory=True)
print("Data Loaders ready")


In [ ]:
class ViolenceClassifier(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()
        bb = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        self.features = bb.features
        self.avgpool  = bb.avgpool
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(960, 512),
            nn.Hardswish(),
            nn.Dropout(dropout/2),
            nn.Linear(512, num_classes),
        )
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = True

    def forward(self, x):
        if x.dim() == 5:
            B,T,C,H,W = x.shape
            x = x.view(B*T,C,H,W)
            f = torch.flatten(self.avgpool(self.features(x)), 1)
            f = f.view(B,T,-1).mean(1)
        else:
            f = torch.flatten(self.avgpool(self.features(x)), 1)
        return self.classifier(f)

model = ViolenceClassifier(dropout=DROPOUT).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW([
    {"params": model.features.parameters(),    "lr": LR * 0.1},
    {"params": model.classifier.parameters(),  "lr": LR},
], weight_decay=1e-4)
scheduler = OneCycleLR(optimizer, max_lr=[LR*0.1, LR],
                       total_steps=EPOCHS*len(train_loader),
                       pct_start=0.1, anneal_strategy="cos")
scaler = torch.cuda.amp.GradScaler()


In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot_loss, tot_acc, n = 0, 0, 0
    all_p, all_l = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for frames, labels in loader:
            frames, labels = frames.to(DEVICE), labels.to(DEVICE)
            with torch.cuda.amp.autocast():
                out  = model(frames)
                loss = criterion(out, labels)
            if train:
                optimizer.zero_grad(); scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update(); scheduler.step()
            bs = labels.size(0)
            tot_loss += loss.item()*bs
            tot_acc  += (out.detach().argmax(1)==labels).sum().item()
            n += bs
            all_p.extend(out.detach().argmax(1).cpu().numpy())
            all_l.extend(labels.cpu().numpy())
    return tot_loss/n, tot_acc/n, np.array(all_p), np.array(all_l)

model.freeze_backbone()
best_acc, patience_cnt = 0, 0
history = {"tr_loss":[],"tr_acc":[],"va_loss":[],"va_acc":[]}


In [ ]:
import shutil, os

for epoch in range(1, EPOCHS+1):
    if epoch == FREEZE_EP+1:
        model.unfreeze_backbone(); print("Backbone unfrozen")

    t0 = time.time()
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, train=True)
    va_loss, va_acc, vp, vl = run_epoch(val_loader, train=False)
    elapsed = time.time()-t0

    history["tr_loss"].append(tr_loss); history["tr_acc"].append(tr_acc)
    history["va_loss"].append(va_loss); history["va_acc"].append(va_acc)

    print(f"Ep {epoch:02d}/{EPOCHS} | {elapsed:.0f}s | "
          f"train {tr_loss:.4f}/{tr_acc*100:.1f}% | "
          f"val {va_loss:.4f}/{va_acc*100:.1f}%", end="")

    if va_acc > best_acc:
        best_acc = va_acc; patience_cnt = 0

        tensors = {k: v.contiguous() for k,v in model.state_dict().items()}
        save_file(tensors, f"{OUTPUT_DIR}/best_model.safetensors",
                  metadata={"backbone":"MobileNetV3-Large","accuracy":f"{best_acc:.4f}"})

        shutil.copy(f"{OUTPUT_DIR}/best_model.safetensors",
                    "/kaggle/working/best_model.safetensors")

        with open("/kaggle/working/history.json", "w") as f:
            json.dump(history, f)

        print(f" saved (best={best_acc*100:.2f}%) → /kaggle/working/best_model.safetensors")
    else:
        patience_cnt += 1
        print(f"  (patience {patience_cnt}/{PATIENCE})")
        if patience_cnt >= PATIENCE:
            print("Early stopping"); break

print(f"\n Best val acc: {best_acc*100:.2f}%")
print(f"   The model saved: /kaggle/working/best_model.safetensors")


In [ ]:
from safetensors.torch import load_file

state = load_file("/kaggle/working/best_model.safetensors")
model.load_state_dict(state)


te_loss, te_acc, te_preds, te_labels = run_epoch(test_loader, train=False)
print(f"\nTest accuracy: {te_acc*100:.2f}%")
print(classification_report(te_labels, te_preds, target_names=["NonViolence","Violence"]))
print("Confusion matrix:")
print(confusion_matrix(te_labels, te_preds))


In [ ]:
import os

files = [
    "/kaggle/working/best_model.safetensors",
    "/kaggle/working/history.json",
]

for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 / 1024
        print(f"{f}  ({size:.1f} MB) — готов к скачиванию")
    else:
        print(f"{f} — не найден")

print("\n ГОТОВО!")
